# RSNA Knee Abnormality Detection — EDA & How Much Can We Trust the Reports?

Quick context before diving in: this is a code competition, the test set has **no report text** (only `train.csv` has `Report`, `test.csv` doesn't). So the reports are only useful as a way to *manufacture more training labels* — not as a model input at inference time.

Only a small subset of studies come with the 12 ground-truth labels. Everything else only has a free-text radiology report. So the real question this notebook tries to answer with numbers, not vibes:

> **If I write a keyword-based labeler and run it on the reports, how close does it get to the real labels?**

If it's decent, we can pseudo-label the huge unlabeled chunk and get way more training signal for free. If it's garbage, we need a smarter (LLM-based) extractor before trusting it.

Spoiler from running this once already: it's not decent yet for every label. Keep reading, the per-label table in section 4 tells you exactly where it breaks and why.


In [ ]:
import os
import glob
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
print(os.listdir(DATA_DIR))


In [ ]:
# Compressed DICOM (JPEG2000 / JPEG Lossless) will fail to decode with plain pydicom.
# The dataset card explicitly says transfer syntaxes are mixed, so install the decoders now
# or you'll get cryptic "unable to decode pixel data" errors halfway through a loop.
!pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm
import pydicom
print(pydicom.__version__)


## 1. Loading the tables

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
train_series = pd.read_csv(f"{DATA_DIR}/train_series.csv")

LABEL_COLS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
              "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
              "Contusion", "Fracture"]

print(train.shape, train_series.shape)
train.head(3)


In [ ]:
# How many studies have the FULL label set vs. report-only?
has_all_labels = train[LABEL_COLS].notna().all(axis=1)
print(f"Studies with the full 12-label set : {has_all_labels.sum()}  ({has_all_labels.mean():.1%})")
print(f"Studies with report text            : {train['Report'].notna().mean():.1%}")

# Don't stop at "all 12 or nothing" -- check per-column availability too.
# It's possible some studies have a handful of labels filled without having all 12,
# which the strict .all(axis=1) filter above would silently throw away.
per_col_available = train[LABEL_COLS].notna().sum().sort_values(ascending=False)
print("\nPer-label non-null counts (may exceed the full-set count above):")
print(per_col_available)
print(f"\nStudies with AT LEAST ONE label filled in: {train[LABEL_COLS].notna().any(axis=1).sum()}")


This ratio is the whole game. If labeled studies are a small slice, the report-derived pseudo-labels below aren't a nice-to-have, they're most of your training signal.

Also worth noting on a first run: the fully-labeled subset came out at **n=58** — that's tiny, every percentage in section 2 below is essentially a coin-flip-sized sample per label, treat it as a rough shape rather than the true population prevalence. If the per-column check above shows meaningfully more non-null values than 58, that means partial labels exist and it's worth building the training set label-by-label (using `.dropna(subset=[label])` per label) instead of requiring all 12 at once — more usable rows per label that way.

## 2. Label prevalence (labeled subset only)

In [ ]:
labeled = train[has_all_labels].copy()

prevalence = labeled[LABEL_COLS].mean().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=prevalence.values, y=prevalence.index, palette="mako")
plt.xlabel("Positive rate")
plt.title(f"Label prevalence on the labeled subset (n={len(labeled)})")
for i, v in enumerate(prevalence.values):
    plt.text(v + 0.005, i, f"{v:.1%}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


Don't assume which labels are rare before looking at this — I guessed wrong the first time I wrote this (expected Fracture/Contusion/Baker's to be the rare ones; on a first run they weren't, MCL came out lowest instead). With n this small the ranking could shuffle on a different random subset too, so:

- Whatever ends up lowest here still needs class weighting / oversampling in training, or the model coasts on easy negatives for AUC.
- Don't treat this ranking as fixed truth about the real (much larger) unlabeled population — it's one small sample.

In [ ]:
corr = labeled[LABEL_COLS].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Label co-occurrence (correlation)")
plt.tight_layout()
plt.show()


Worth checking: do the three OA compartments (Medial/Lateral/PF) correlate with each other, and how strongly? Clinically they often co-occur in advanced disease, but on n=58 don't expect a clean signal — read the actual numbers in the heatmap rather than assuming "strong" or "weak" going in.

## 3. Series-level structure — what are we actually feeding the model

In [ ]:
series_per_study = train_series.groupby("StudyInstanceUID").size()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(series_per_study, bins=20, ax=axes[0], color="#4c72b0")
axes[0].set_title("Series per study")
axes[0].set_xlabel("# series")

sns.countplot(data=train_series, x="Anatomical_Plane", ax=axes[1],
              order=train_series["Anatomical_Plane"].value_counts().index, color="#55a868")
axes[1].set_title("Anatomical plane")

combo = train_series.groupby(["Fluid_Sensitive", "Fat_Suppression"]).size().reset_index(name="count")
combo["combo"] = combo.apply(lambda r: f"FS={r.Fluid_Sensitive} FatSat={r.Fat_Suppression}", axis=1)
sns.barplot(data=combo, x="combo", y="count", ax=axes[2], color="#c44e52")
axes[2].set_title("Fluid-sensitive x Fat-sat combos")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

print(series_per_study.describe())
print("\nFluid_Sensitive x Fat_Suppression combo counts (check this before assuming anything):")
print(combo[["Fluid_Sensitive", "Fat_Suppression", "count"]])


Why this matters for the model, not just curiosity: the dataset card is basically handing you a clinical prior for free.

- Meniscus / ligament tears read best on fluid-sensitive sagittal/coronal sequences.
- Effusion, contusion, and fracture pop on fluid-sensitive + fat-suppressed (STIR-like) sequences.
- OA (cartilage loss, joint space narrowing) is more of a structural/T1-ish read, less dependent on fluid sensitivity.

Instead of pooling all series with equal weight into a study-level prediction, gate the series-level attention using `Fluid_Sensitive` / `Fat_Suppression` / `Anatomical_Plane` as auxiliary features.

One thing to double check rather than assume: on a first run, only two combos showed up (FS=0/FatSat=0 and FS=1/FatSat=1) — the two flags moved together perfectly, no FS=1/FatSat=0 or FS=0/FatSat=1 cases. That's not typical of every real knee protocol, so worth confirming with the printed counts above rather than trusting the bar chart shape alone — it's possible these two fields are simplified/derived rather than fully independent in this dataset.

## 4. Reports: languages, and can a keyword labeler actually recover the ground truth?

In [ ]:
# Multiple reporting sites -> multiple languages, per the dataset card.
# Cheap language signal without extra dependencies: character set + a handful of
# language-specific stopwords/markers. Good enough to see the split, not meant to be exact.
def rough_lang_guess(text):
    if not isinstance(text, str) or len(text) < 5:
        return "unknown"
    t = text.lower()
    markers = {
        "en": [" the ", " and ", " with ", " no ", " findings"],
        "es": [" el ", " la ", " con ", " sin ", " hallazgos"],
        "fr": [" le ", " la ", " avec ", " sans ", " pas de"],
        "de": [" der ", " die ", " das ", " mit ", " kein "],
        "pt": [" o ", " a ", " com ", " sem ", " achados"],
    }
    scores = {lang: sum(t.count(m) for m in ms) for lang, ms in markers.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "other/unknown"

sample = train["Report"].dropna().sample(min(3000, train["Report"].notna().sum()), random_state=0)
lang_counts = sample.apply(rough_lang_guess).value_counts(normalize=True)
lang_counts.plot(kind="bar", color="#8172b2", figsize=(7,4))
plt.title("Rough language mix of a report sample (heuristic, not a real detector)")
plt.ylabel("share")
plt.tight_layout()
plt.show()


If `other/unknown` + non-English markers together add up to a large share (they did on a first run — well over half the sample wasn't confidently English), that's the single most important number in this notebook: **a pure English keyword list is structurally capped, no amount of tuning the phrase list fixes a language mismatch.** Confirm the language mix of the labeled n=58 subset specifically too (not just the full sample) — the precision/recall table below is only trustworthy for the population it was measured on.

In [ ]:
# A minimal, deliberately simple keyword labeler. The point isn't to be clever,
# it's to get a *measured* baseline precision/recall so pseudo-labeling decisions
# are based on numbers instead of assuming report text = ground truth.
KEYWORDS = {
    "ACL":               ["acl tear", "acl rupture", "anterior cruciate ligament tear",
                           "acl injury", "torn acl"],
    "MCL":               ["mcl tear", "mcl sprain", "medial collateral ligament"],
    "Medial Meniscus":   ["medial meniscus tear", "medial meniscal tear", "medial meniscus injury"],
    "Lateral Meniscus":  ["lateral meniscus tear", "lateral meniscal tear", "lateral meniscus injury"],
    "Medial OA":         ["medial compartment osteoarthritis", "medial osteoarthritis",
                           "medial joint space narrowing"],
    "Lateral OA":        ["lateral compartment osteoarthritis", "lateral osteoarthritis"],
    "PF OA":             ["patellofemoral osteoarthritis", "patellofemoral compartment osteoarthritis"],
    "Effusion":          ["joint effusion", "knee effusion", "suprapatellar effusion"],
    "Synovitis":         ["synovitis", "synovial thickening", "synovial proliferation"],
    "Baker's":           ["baker's cyst", "bakers cyst", "popliteal cyst"],
    "Contusion":         ["bone contusion", "bone bruise", "osseous contusion", "marrow edema"],
    "Fracture":          ["fracture", "cortical break"],
}
NEGATION_WINDOW = ["no ", "without ", "negative for ", "no evidence of "]

def keyword_predict(report, label):
    if not isinstance(report, str):
        return 0
    text = report.lower()
    for kw in KEYWORDS[label]:
        idx = text.find(kw)
        if idx == -1:
            continue
        window = text[max(0, idx - 25): idx]
        if any(neg in window for neg in NEGATION_WINDOW):
            continue
        return 1
    return 0

from sklearn.metrics import precision_score, recall_score, f1_score

rows = []
eval_df = labeled.dropna(subset=["Report"])
for label in LABEL_COLS:
    y_true = eval_df[label].astype(int)
    y_pred = eval_df["Report"].apply(lambda r: keyword_predict(r, label))
    rows.append({
        "label": label,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "n_positive": int(y_true.sum()),
        "n_predicted_positive": int(y_pred.sum()),
    })

report_quality = pd.DataFrame(rows).sort_values("f1", ascending=False)
report_quality


In [ ]:
plt.figure(figsize=(9, 5))
melted = report_quality.melt(id_vars="label", value_vars=["precision", "recall", "f1"])
sns.barplot(data=melted, x="value", y="label", hue="variable")
plt.title("Keyword labeler vs. ground truth, per label")
plt.tight_layout()
plt.show()


Read the actual table before pseudo-labeling anything — don't assume "reports = free labels" just because reports exist:

- **A label with `n_predicted_positive` near 0** means the keyword list didn't fire almost at all for that label on this sample. Zero precision AND zero recall together is a sign of a complete miss, not just a weak one — usually because the exact English phrases in the list don't appear (wrong language, or clinicians phrasing it completely differently, e.g. "joint space narrowing" instead of the word "osteoarthritis").
- **High precision, low recall** → safe to use as one-sided pseudo-*positive* labeling only (trust the hits, don't assume a miss means negative — recall this low means plenty of true positives get missed and would be mislabeled negative).
- **Both low** → shortlist that label for a proper LLM-based / multilingual extractor before using it for anything.

This table is the actual deliverable of this section: a *per-label* trust score for weak supervision, measured, not assumed.

## 5. What a study actually looks like — slice grid across planes

In [ ]:
def load_series_slices(study_uid, series_uid, max_slices=9):
    series_dir = f"{DATA_DIR}/train_series/{study_uid}/{series_uid}"
    files = sorted(glob.glob(f"{series_dir}/*.dcm"))
    if not files:
        return []
    step = max(1, len(files) // max_slices)
    picked = files[::step][:max_slices]
    slices = []
    for fp in picked:
        try:
            ds = pydicom.dcmread(fp)
            arr = ds.pixel_array.astype(np.float32)
            arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
            slices.append(arr)
        except Exception as e:
            print(f"skipped {fp}: {e}")
    return slices

# Pick one study that has multiple series with different planes, for a good comparison
example_study = train_series["StudyInstanceUID"].value_counts().index[0]
example_series = train_series[train_series["StudyInstanceUID"] == example_study]
print(example_series[["SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]])


In [ ]:
fig, axes = plt.subplots(1, min(4, len(example_series)), figsize=(16, 4))
if len(example_series) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, example_series.iloc[:4].iterrows()):
    slices = load_series_slices(example_study, row["SeriesInstanceUID"], max_slices=1)
    if slices:
        ax.imshow(slices[0], cmap="gray")
    ax.set_title(f"{row['Anatomical_Plane']}\nFS={row['Fluid_Sensitive']} FatSat={row['Fat_Suppression']}",
                 fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


If any of those panels came back blank with a decode error in the printed log above, that's the compressed-transfer-syntax issue mentioned at the top — double check the `pylibjpeg`/`gdcm` install actually took before assuming your data loader is broken.

## 6. Takeaways

- Report-only studies vastly outnumber fully labeled ones — pseudo-labeling isn't optional here, it's most of the usable signal, but section 4's table shows it's not equally trustworthy across all 12 labels.
- Check per-column label availability, not just the strict "all 12 present" filter — there may be more usable rows hiding per-label.
- `Fluid_Sensitive` / `Fat_Suppression` / `Anatomical_Plane` are a free clinical prior for weighting series in aggregation — but verify the combo counts before assuming they behave the way a textbook knee protocol would.
- Language mix matters more than phrase-list cleverness. Check it before investing time in expanding an English-only keyword list.
- Don't skip the DICOM decoder install. It's a five-second `pip install` that saves an afternoon of "why is my dataloader silently dropping half the studies."

If this was useful, an upvote helps it reach more people starting out on this comp — and if you spot a label where the keyword list is obviously missing something (Synovitis and Contusion phrasing especially, and non-English OA terms like "artrosis" / "arthrose" / "Gonarthrose"), drop it in the comments, happy to fold it into a v2 pass on the labeler.
